# Small Language Model (SLM)
A decoder-only Transformer, as in GPT (Radford et al., 2018-2019), built with the Llama recipe (Touvron et al., 2023). "Small" = few enough parameters to run on-device: Phi, Gemma-2B, Qwen-0.5B, TinyLlama, SmolLM. Ours is ~5M params, character level.

**Idea:** drop recurrence. Every position attends directly to all previous positions (causal mask), so training on a whole sequence happens in parallel, and there's no fixed-size bottleneck state.

### SLM vs RNN/LSTM/GRU vs original Transformer
| | RNN / LSTM / GRU | Original Transformer (2017) | SLM (Llama-style decoder) |
|---|---|---|---|
| Parts | recurrent cell | encoder + decoder | **decoder only** |
| Access to past | through fixed-size state $h_t$ | attention | causal attention to every past token |
| Training over time | sequential | parallel | parallel |
| Position | implicit (loop order) | sin/cos added to input | **RoPE** rotates $q, k$ |
| Norm | - | Post-LN, LayerNorm | **Pre-norm, RMSNorm** |
| FFN | - | ReLU MLP | **SwiGLU** |
| Generation cost / token | $O(1)$ | $O(T)$ | $O(T)$ (KV cache avoids recompute, still attends to $T$) |

Shape flow: `idx (B, T) -> embed (B, T, D) -> [RMSNorm -> causal attn(RoPE) -> + -> RMSNorm -> SwiGLU -> +] x L -> RMSNorm -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - tiny Shakespeare (character level)
~1.1M characters, vocab = 65 unique characters. Each training example is a random chunk of `block_size` characters; the target is the same chunk shifted by one (predict the next character).

The same data and loss are used in `rnn`, `lstm`, `gru`, `slm`, `moe`, `ssm`, so their val losses can be compared.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 128   # T: characters per sequence
batch_size = 32   # B

def get_batch(split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab size:", vocab_size, "| train chars:", len(train_data), "| val chars:", len(val_data))

## 2. Modern decoder pieces (Llama-style)
Three swaps relative to the original Transformer, used by nearly every current SLM/LLM (Llama, Qwen, Phi, Gemma, Mistral):

### Formula: RMSNorm (replaces LayerNorm)
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{D}\sum_{j=1}^{D} x_j^2 + \epsilon}} \odot \gamma$$
- $x \in \mathbb{R}^{D}$: one token's features
- no mean subtraction and no bias $\beta$ (cheaper; works as well as LayerNorm)
- $\gamma \in \mathbb{R}^{D}$: learned scale (`self.weight`), init 1
- $\epsilon$: $10^{-6}$, avoids division by zero

### Formula: Rotary position embedding (RoPE, replaces added positional encodings)
$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}, \qquad \theta_i = 10000^{-2i/d}$$
- $q$: a query (or key) vector of one head, dim $d$ (= head_dim)
- $m$: the token's position
- each pair of dims $(2i, 2i+1)$ is rotated by angle $m\theta_i$; low $i$ rotates fast, high $i$ slowly (same frequencies as sinusoidal PE)
- applied to $q$ and $k$ only, so $q'_m \cdot k'_n$ depends on the **relative** offset $m - n$

### Formula: Causal attention
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d}} + M\right)V, \qquad M_{mn} = \begin{cases} 0 & n \le m \\ -\infty & n > m \end{cases}$$
- $Q, K, V \in \mathbb{R}^{T \times d}$ per head, $d$ = head_dim
- $M$: causal mask; $-\infty$ becomes 0 after softmax, so token $m$ can't see future tokens
- training is still parallel: all $T$ next-token predictions in one pass

### Formula: SwiGLU FFN (replaces ReLU/GELU MLP)
$$\text{FFN}(x) = W_2\left(\text{SiLU}(W_1 x) \odot W_3 x\right), \qquad \text{SiLU}(u) = u \cdot \sigma(u)$$
- $W_1, W_3 \in \mathbb{R}^{H \times D}$: gate and up projections; $W_2 \in \mathbb{R}^{D \times H}$: down projection
- $\odot$: element-wise; one branch learns *what* to pass, the other *how much* (a learned gate)
- $H = \frac{8}{3}D$ (rounded to 64) keeps params equal to a $4D$ two-matrix MLP

In [ ]:
class RMSNorm(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # gamma

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope_cache(T, head_dim, base=10000):
    # -> cos, sin each (T, head_dim/2)
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)  # (head_dim/2,) theta_i
    angles = torch.arange(T).float()[:, None] * freqs[None]                  # (T, head_dim/2) m * theta_i
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    # x (B, heads, T, hd) -> (B, heads, T, hd); rotate each pair (x_2i, x_2i+1)
    x1, x2 = x[..., 0::2], x[..., 1::2]                                      # each (B, heads, T, hd/2)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class CausalSelfAttention(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)  # (B, T, 3, heads, hd)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                                # each (B, heads, T, hd)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)             # position enters here, not at the input

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5              # (B, heads, T, T)
        mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()   # (T, T) lower triangle = allowed
        att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)

        out = (att @ v).transpose(1, 2).reshape(B, T, D)                    # concat heads -> (B, T, D)
        return self.proj(out)


class SwiGLU(nn.Module):
    # (..., D) -> (..., D)
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)  # gate
        self.w3 = nn.Linear(dim, hidden, bias=False)  # up
        self.w2 = nn.Linear(hidden, dim, bias=False)  # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # (..., H) -> (..., D)


def ffn_hidden(dim):
    return 64 * ((8 * dim // 3 + 63) // 64)  # 8/3 * D rounded up to a multiple of 64

## 3. Decoder block and model
Pre-norm residual block, same as ViT but with RMSNorm/SwiGLU and a causal mask. The output head shares its weights with the token embedding (**weight tying**).

### Formula: Pre-norm block
$$x' = x + \text{Attn}(\text{RMSNorm}(x)), \qquad x_{out} = x' + \text{FFN}(\text{RMSNorm}(x'))$$
- $x \in \mathbb{R}^{T \times D}$: token features
- the residual stream $x$ is never normalized in place, so gradients flow straight through $L$ blocks

### Formula: Weight tying
$$\text{logits} = \text{RMSNorm}(x_L)\, E^T$$
- $E \in \mathbb{R}^{V \times D}$: token embedding matrix, reused as the output layer
- saves $V \times D$ params and ties "reading" and "writing" a token to the same vector

In [ ]:
class Block(nn.Module):
    # (B, T, D) -> (B, T, D)
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads)
        self.norm2 = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_hidden(dim))

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)  # (B, T, D)
        x = x + self.ffn(self.norm2(x))             # (B, T, D)
        return x


class SLM(nn.Module):
    # idx (B, T) -> logits (B, T, V)
    def __init__(self, vocab_size, block_size, dim=256, depth=6, num_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, num_heads) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight           # weight tying
        nn.init.normal_(self.embed.weight, std=0.02)
        cos, sin = rope_cache(block_size, dim // num_heads)
        self.register_buffer("cos", cos)                # (block_size, hd/2)
        self.register_buffer("sin", sin)

    def forward(self, idx, state=None):
        T = idx.shape[1]
        x = self.embed(idx)                             # (B, T, D), no position added here
        for block in self.blocks:
            x = block(x, self.cos[:T], self.sin[:T])    # (B, T, D)
        return self.head(self.norm(x)), None            # (B, T, V); None = no recurrent state

model = SLM(vocab_size, block_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(model(torch.zeros(2, 10, dtype=torch.long, device=device))[0].shape)

## 4. Training
Same objective as the RNNs, but all $T$ positions are computed in one parallel pass.

### Formula: Next-token cross-entropy
$$\mathcal{L} = -\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T} \log p_\theta(x_{t+1} \mid x_{\le t})$$
- $B, T$: batch size and sequence length
- $x_{t+1}$: the true next character
- $p_\theta(\cdot \mid x_{\le t})$: softmax over the vocab from the logits at position $t$
- a model that guesses uniformly scores $\log 65 \approx 4.17$; lower is better

### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1, \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: all parameter gradients concatenated into one vector
- $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales the whole gradient, keeps its direction

In [ ]:
max_iters = 3000
eval_every = 500
lr = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def loss_fn(model, x, y):
    logits, _ = model(x)                                              # (B, T, V)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # (B*T, V) vs (B*T,) -> scalar

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    x, y = get_batch("train")
    loss = loss_fn(model, x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 5. Generation
Autoregressive: append the sampled char and run again. For simplicity we recompute the whole context each step; real SLMs keep a **KV cache** (the $K, V$ of past tokens) so each step only computes the new token.

### Formula: Sampling with temperature
$$p_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}, \qquad x_{t+1} \sim \text{Categorical}(p)$$
- $z_i$: logit of character $i$
- $\tau$: temperature (0.8); $\tau < 1$ sharpens (safer), $\tau > 1$ flattens (more random), $\tau \to 0$ = greedy argmax

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)           # (1, t)
    for _ in range(n):
        logits, _ = model(idx[:, -block_size:])                   # re-reads the last block_size chars each step (no KV cache)
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)    # (1, V)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)  # (1, t+1)
    return decode(idx[0].tolist())

print(generate(model))

## Notes
- Expect a clearly lower val loss than the RNN/LSTM/GRU notebooks with a similar training budget.
- Real SLMs are the same architecture at 0.5-3B params, subword (BPE) tokens and trillions of training tokens. Small models get strong mostly through **data quality** and distillation from bigger models, not architecture changes.
- The FFN holds most of the parameters. **MoE** (next) replaces it with many expert FFNs and routes each token to a few of them.